In [1]:
# ============================================
# Cell 1) 라이브러리 / 공통 설정
# ============================================
import re
import numpy as np
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine, text

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 60)
print("[OK] imports loaded")

[OK] imports loaded


In [2]:
# ============================================
# Cell 2) DB 설정 / 엔진 생성
# ============================================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "c1_fct_detail"
TARGET_TABLE  = "fct_detail"

TARGET_END_DAY = "2025-11-13"
TARGET_REMARK  = "Non-PD"

def get_engine(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?connect_timeout=5"
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine(DB_CONFIG)
print("[OK] engine created")

[OK] engine created


In [3]:
# ============================================
# Cell 3) 데이터 로딩 (end_day=2025-11-13, remark=Non-PD)
# - file_path 제외
# ============================================
sql = text(f"""
SELECT
    barcode_information,
    remark,
    end_day,
    end_time,
    test_time,
    contents,
    test_ct
FROM {TARGET_SCHEMA}.{TARGET_TABLE}
WHERE end_day::date = :end_day
  AND remark = :remark
ORDER BY
    barcode_information ASC,
    end_time::time ASC,
    test_time::time ASC NULLS LAST,
    contents ASC
""")

df = pd.read_sql(
    sql,
    engine,
    params={
        "end_day": TARGET_END_DAY,
        "remark": TARGET_REMARK
    }
)

print(f"[OK] loaded rows={len(df)}")
df.head(30)

[OK] loaded rows=708318


,barcode_information,remark,end_day,end_time,test_time,contents,test_ct
0,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:20.66,MES 이전공정 체크 SKIP,NaN
1,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.50,START :: DO SET,1.84
2,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.51,DO_SET Value :: 0900800080001000,0.01
3,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.53,테스트 결과 : OK,0.02
4,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.54,START :: LOAD_A SET CC MODE,0.01
5,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.55,Return Value :: OK,0.01
6,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.56,테스트 결과 : OK,0.01
7,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.57,START :: LOAD_A SET RANG,0.01
8,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.72,Return Value :: OK,0.15
9,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.73,테스트 결과 : OK,0.01


In [4]:
# ============================================
# Cell 4) group 생성 (재시험 분리 핵심)
# - group 키: (barcode_information, end_day, end_time)
# - end_time이 다르면 무조건 다른 group
# ============================================
df_base = df.copy()

df_base["barcode_information"] = df_base["barcode_information"].astype(str)
df_base["remark"] = df_base["remark"].astype(str)
df_base["end_day"] = pd.to_datetime(df_base["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df_base["end_time"] = df_base["end_time"].astype(str)

# test_time 정렬용
df_base["test_time_str"] = df_base["test_time"].astype(str)
df_base["_tt"] = pd.to_datetime(df_base["test_time_str"], format="%H:%M:%S.%f", errors="coerce")

# (중요) group 부여: barcode+end_day+end_time
df_base["group"] = (
    df_base.groupby(["barcode_information", "end_day", "end_time"], sort=False)
           .ngroup()
           .add(1)
           .astype(int)
)

# 우선 기본 정렬
df_base = df_base.sort_values(
    ["group", "_tt", "test_time_str"],
    ascending=[True, True, True],
    na_position="last"
).reset_index(drop=True)

print(f"[OK] groups={df_base['group'].nunique()} rows={len(df_base)}")
df_base.head(30)

[OK] groups=3354 rows=708318


,barcode_information,remark,end_day,end_time,test_time,contents,test_ct,test_time_str,_tt,group
0,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:20.66,MES 이전공정 체크 SKIP,NaN,08:23:20.66,1900-01-01 08:23:20.660,1
1,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.50,START :: DO SET,1.84,08:23:22.50,1900-01-01 08:23:22.500,1
2,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.51,DO_SET Value :: 0900800080001000,0.01,08:23:22.51,1900-01-01 08:23:22.510,1
3,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.53,테스트 결과 : OK,0.02,08:23:22.53,1900-01-01 08:23:22.530,1
4,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.54,START :: LOAD_A SET CC MODE,0.01,08:23:22.54,1900-01-01 08:23:22.540,1
5,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.55,Return Value :: OK,0.01,08:23:22.55,1900-01-01 08:23:22.550,1
6,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.56,테스트 결과 : OK,0.01,08:23:22.56,1900-01-01 08:23:22.560,1
7,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.57,START :: LOAD_A SET RANG,0.01,08:23:22.57,1900-01-01 08:23:22.570,1
8,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.72,Return Value :: OK,0.15,08:23:22.72,1900-01-01 08:23:22.720,1
9,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-13,08:23:40,08:23:22.73,테스트 결과 : OK,0.01,08:23:22.73,1900-01-01 08:23:22.730,1


In [5]:
# ============================================
# Cell 5) group 필터링 + 그룹 내 정렬 규칙 적용
# 6) "START :: MES 이전공정 체크 SKIP" 포함 group 제외
# 7) group 내에 "3~" 시작 + test_ct NULL 행이 없으면 제외
# 8) 그 "3~"+test_ct NULL 행을 group 첫행으로 강제
# 9) 나머지는 test_time 오름차순
# ============================================

SKIP_TEXT = "START :: MES 이전공정 체크 SKIP"

def is_three_row(s: str) -> bool:
    return str(s).startswith("3~")

def is_null_like(x) -> bool:
    # DB에서 NULL 또는 문자열 'NULL' 등 방어
    if x is None:
        return True
    if isinstance(x, float) and pd.isna(x):
        return True
    s = str(x).strip().lower()
    return (s == "" or s == "none" or s == "null" or s == "nan")

kept = []
for gid, g in df_base.groupby("group", sort=False):
    # (6) SKIP 있으면 제외
    if g["contents"].astype(str).str.contains(SKIP_TEXT, na=False).any():
        continue

    # (7) "3~" + test_ct NULL 존재해야
    three_mask = g["contents"].astype(str).apply(is_three_row)
    if not three_mask.any():
        continue

    # "3~" 중 test_ct NULL인 행만
    three_null = g.loc[three_mask].copy()
    three_null["_is_null_ct"] = three_null["test_ct"].apply(is_null_like)
    three_null = three_null[three_null["_is_null_ct"]]

    if len(three_null) == 0:
        continue

    # group 내에서 "3~"+ctNULL 첫 행 결정: test_time 기준으로 가장 빠른 것
    three_null = three_null.sort_values(["_tt", "test_time_str"], ascending=[True, True], na_position="last")
    first_idx = three_null.index[0]

    gg = g.copy()
    gg["_first_flag"] = (gg.index == first_idx).astype(int)

    # (8) first_flag(1) 최우선, (9) 이후 test_time 오름차순
    gg = gg.sort_values(
        ["_first_flag", "_tt", "test_time_str"],
        ascending=[False, True, True],
        na_position="last"
    )

    kept.append(gg)

df2 = pd.concat(kept, ignore_index=True) if kept else df_base.iloc[0:0].copy()

# 정리
df2.drop(columns=["_first_flag"], inplace=True, errors="ignore")

print(f"[OK] after filter groups={df2['group'].nunique()} rows={len(df2)}")
df2.head(50)

[OK] after filter groups=3334 rows=705851


,barcode_information,remark,end_day,end_time,test_time,contents,test_ct,test_time_str,_tt,group
0,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.28,3~192.168.108.154~FCT~PPIDBA1WJ25300500277PC3T...,NaN,17:17:57.28,1900-01-01 17:17:57.280,20
1,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.29,00664~192.168.108.154~FCT~PPIDBA1WJ2530050027...,0.01,17:17:57.29,1900-01-01 17:17:57.290,20
2,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.91,START :: DO SET,1.62,17:17:58.91,1900-01-01 17:17:58.910,20
3,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.93,DO_SET Value :: 0900800080001000,0.02,17:17:58.93,1900-01-01 17:17:58.930,20
4,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.94,테스트 결과 : OK,0.01,17:17:58.94,1900-01-01 17:17:58.940,20
5,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.95,START :: LOAD_A SET CC MODE,0.01,17:17:58.95,1900-01-01 17:17:58.950,20
6,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.96,Return Value :: OK,0.01,17:17:58.96,1900-01-01 17:17:58.960,20
7,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.98,테스트 결과 : OK,0.02,17:17:58.98,1900-01-01 17:17:58.980,20
8,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:59.47,START :: LOAD_A SET RANG,0.49,17:17:59.47,1900-01-01 17:17:59.470,20
9,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:59.62,Return Value :: OK,0.15,17:17:59.62,1900-01-01 17:17:59.620,20


In [6]:
# ============================================
# Cell 6) group 내 첫 행(=3~ + ct NULL) 검증(선택)
# - 각 group 첫 행이 조건을 만족하는지 샘플 확인
# ============================================
tmp = df2.groupby("group", sort=False).head(1).copy()
tmp["is_3"] = tmp["contents"].astype(str).str.startswith("3~", na=False)
tmp["is_null_ct"] = tmp["test_ct"].apply(lambda x: (x is None) or (isinstance(x, float) and pd.isna(x)) or str(x).strip().lower() in ["", "none", "null", "nan"])
print("[INFO] first-row condition check (top 30):")
tmp[["group","barcode_information","end_time","test_time","contents","test_ct","is_3","is_null_ct"]].head(30)

[INFO] first-row condition check (top 30):


,group,barcode_information,end_time,test_time,contents,test_ct,is_3,is_null_ct
0,20,BA1WJ25300500277PC3T-14F014-AC,17:18:18,17:17:57.28,3~192.168.108.154~FCT~PPIDBA1WJ25300500277PC3T...,NaN,True,True
212,21,BA1WJ25300500289PC3T-14F014-AC,17:19:08,17:18:44.86,3~192.168.108.153~FCT~PPIDBA1WJ25300500289PC3T...,NaN,True,True
424,22,BA1WJ25300500421PC3T-14F014-AC,17:14:50,17:14:26.86,3~192.168.108.153~FCT~PPIDBA1WJ25300500421PC3T...,NaN,True,True
638,23,BA1WJ25301501723PC3T-14F014-AC,17:17:42,17:17:21.28,3~192.168.108.154~FCT~PPIDBA1WJ25301501723PC3T...,NaN,True,True
849,24,BA1WJ25301501731PC3T-14F014-AC,17:19:20,17:18:58.91,3~192.168.108.154~FCT~PPIDBA1WJ25301501731PC3T...,NaN,True,True
1061,25,BA1WJ25301502816PC3T-14F014-AC,17:18:03,17:17:39.68,3~192.168.108.153~FCT~PPIDBA1WJ25301502816PC3T...,NaN,True,True
1273,26,BA1WJ25301503547PC3T-14F014-AC,17:16:26,17:16:02.79,3~192.168.108.153~FCT~PPIDBA1WJ25301503547PC3T...,NaN,True,True
1487,27,BA1WJ25301503674PC3T-14F014-AC,17:16:59,17:16:35.22,3~192.168.108.153~FCT~PPIDBA1WJ25301503674PC3T...,NaN,True,True
1701,28,BA1WJ25301504100PC3T-14F014-AC,17:17:10,17:16:48.85,3~192.168.108.154~FCT~PPIDBA1WJ25301504100PC3T...,NaN,True,True
1914,29,BA1WJ25301504269PC3T-14F014-AC,17:16:37,17:16:16.28,3~192.168.108.154~FCT~PPIDBA1WJ25301504269PC3T...,NaN,True,True


In [7]:
# ============================================
# Cell 7) from_to_test_ct 생성
# - 기준: 각 group의 "첫 행 test_time"
# - 대상: contents == '테스트 결과 : OK' or '테스트 결과 : NG'
# - 계산: (해당 행 test_time) - (기준 test_time)  [초(float)]
# ============================================

def test_time_to_seconds(x):
    """HH:MM:SS.fff -> 초(float). 실패 시 NA."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return pd.NA
    m = re.match(r"^\s*(\d+):(\d+):(\d+)(?:\.(\d+))?\s*$", s)
    if not m:
        return pd.NA
    hh, mm, ss = int(m.group(1)), int(m.group(2)), int(m.group(3))
    frac = m.group(4)
    base = hh * 3600 + mm * 60 + ss
    if frac is None:
        return float(base)
    return float(base) + float("0." + frac)

df3 = df2.copy()

df3["_tt_sec"] = df3["test_time"].apply(test_time_to_seconds)
df3["_base_sec"] = df3.groupby("group")["_tt_sec"].transform("first")

is_result = df3["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df3["from_to_test_ct"] = pd.NA
mask = is_result & df3["_tt_sec"].notna() & df3["_base_sec"].notna()
df3.loc[mask, "from_to_test_ct"] = (df3.loc[mask, "_tt_sec"] - df3.loc[mask, "_base_sec"]).astype(float)

df3.drop(columns=["_tt_sec", "_base_sec"], inplace=True, errors="ignore")

print(f"[OK] from_to_test_ct filled rows = {df3['from_to_test_ct'].notna().sum()}")
df3.head(30)

[OK] from_to_test_ct filled rows = 222512


,barcode_information,remark,end_day,end_time,test_time,contents,test_ct,test_time_str,_tt,group,from_to_test_ct
0,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.28,3~192.168.108.154~FCT~PPIDBA1WJ25300500277PC3T...,NaN,17:17:57.28,1900-01-01 17:17:57.280,20,<NA>
1,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.29,00664~192.168.108.154~FCT~PPIDBA1WJ2530050027...,0.01,17:17:57.29,1900-01-01 17:17:57.290,20,<NA>
2,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.91,START :: DO SET,1.62,17:17:58.91,1900-01-01 17:17:58.910,20,<NA>
3,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.93,DO_SET Value :: 0900800080001000,0.02,17:17:58.93,1900-01-01 17:17:58.930,20,<NA>
4,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.94,테스트 결과 : OK,0.01,17:17:58.94,1900-01-01 17:17:58.940,20,1.66
5,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.95,START :: LOAD_A SET CC MODE,0.01,17:17:58.95,1900-01-01 17:17:58.950,20,<NA>
6,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.96,Return Value :: OK,0.01,17:17:58.96,1900-01-01 17:17:58.960,20,<NA>
7,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.98,테스트 결과 : OK,0.02,17:17:58.98,1900-01-01 17:17:58.980,20,1.7
8,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:59.47,START :: LOAD_A SET RANG,0.49,17:17:59.47,1900-01-01 17:17:59.470,20,<NA>
9,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:59.62,Return Value :: OK,0.15,17:17:59.62,1900-01-01 17:17:59.620,20,<NA>


In [8]:
# ============================================
# Cell 8) OK/NG 순번(1~67) + test_contents 매핑 + 출력
# - 핵심: group 기준 cumcount (재시험 분리 유지)
# - 출력 컬럼 순서 고정
# ============================================

# 1~67 매핑
MAP_1_67 = {
  1:"0.00_d_sig_val_090_set",
  2:"0.00_load_a_cc_set",
  3:"0.00_dmm_c_rng_set",
  4:"0.00_load_c_cc_set",
  5:"0.00_dmm_c_rng_set",
  6:"0.00_dmm_regi_set",
  7:"0.00_dmm_regi_ac_0.6_set",
  8:"1.00_mini_b_short_check",
  9:"1.01_usb_a_short_check",
  10:"1.02_usb_c_short_check",
  11:"1.03_d_sig_val_000_set",
  12:"1.03_dmm_regi_set",
  13:"1.03_dmm_regi_ac_0.6_set",
  14:"1.03_pin12_short_check",
  15:"1.04_pin23_short_check",
  16:"1.05_pin34_short_check",
  17:"1.06_dmm_dc_v_set",
  18:"1.06_dmm_ac_0.6_set",
  19:"1.06_dmm_c_set",
  20:"1.06_load_a_sensing_on",
  21:"1.06_load_c_sensing_on",
  22:"1.06_ps_16.5v_set",
  23:"1.06_ps_on",
  24:"1.06_dmm_ac_0.6_set",
  25:"1.06_input_16.5v",
  26:"1.07_idle_c_check",
  27:"1.08_fw_ver_check",
  28:"1.09_chip_id_check",
  29:"1.10_dmm_3c_rng_set",
  30:"1.10_load_a_5.5c_set",
  31:"1.10_load_a_on",
  32:"1.10_usb_a_v_check",
  33:"1.11_usb_a_c_check",
  34:"1.12_usb_c_v_check",
  35:"1.13_load_a_off",
  36:"1.13_load_c_5.5c_set",
  37:"1.13_load_c_on",
  38:"1.13_overcurr_usb_c_v",
  39:"1.14_overcurr_usb_c_c",
  40:"1.15_usb_a_v",
  41:"1.16_load_c_off",
  42:"1.16_dut_reset",
  43:"1.16_cc_aside_check",
  44:"1.17_cc_bside_check",
  45:"1.18_load_a_2.4c_set",
  46:"1.18_load_c_3c_set",
  47:"1.18_load_a_on",
  48:"1.18_load_c_on",
  49:"1.18_usb_a_v_check",
  50:"1.19_usb_a_c_check",
  51:"1.20_usb_c_v_check",
  52:"1.21_usb_c_c_check",
  53:"1.22_load_a_off",
  54:"1.22_load_c_off",
  55:"1.22_usb_c_carplay",
  56:"1.23_usb_a_carplay",
  57:"1.24_usb_c_pm1",
  58:"1.25_usb_c_pm2",
  59:"1.26_usb_c_pm3",
  60:"1.27_usb_c_pm4",
  61:"1.28_usb_a_pm1",
  62:"1.29_usb_a_pm2",
  63:"1.30_usb_a_pm3",
  64:"1.31_usb_a_pm4",
  65:"1.32_dmm_ac_0.6_set",
  66:"1.32_dmm_c_rng_set",
  67:"1.32_dark_curr_check",
}

df4 = df3.copy()

is_okng = df4["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df4["okng_seq"] = pd.NA
df4.loc[is_okng, "okng_seq"] = (
    df4.loc[is_okng]
       .groupby("group")
       .cumcount()
       .add(1)
       .astype(int)
)

df4["test_contents"] = pd.NA
df4.loc[is_okng, "test_contents"] = df4.loc[is_okng, "okng_seq"].map(MAP_1_67)

# 출력 컬럼 순서 고정
final_cols = [
    "group",
    "barcode_information",
    "remark",
    "end_day",
    "end_time",
    "test_time",
    "contents",
    "test_contents",
    "test_ct",
    "from_to_test_ct",
    "okng_seq",
]
df4_out = df4[final_cols].copy()

# 진단: group별 OK/NG 개수 (정상은 <= 67, 완전하면 67)
cnt = df4.loc[is_okng].groupby("group")["okng_seq"].max().fillna(0).astype(int)
print("[INFO] OK/NG count per group (top 20):")
display(cnt.sort_values(ascending=False).head(20))

df4_out.head(500)

[INFO] OK/NG count per group (top 20):


C:\Users\user\AppData\Local\Temp\ipykernel_6424\3174924326.py:111: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cnt = df4.loc[is_okng].groupby("group")["okng_seq"].max().fillna(0).astype(int)


group
20      67
2237    67
2227    67
2228    67
2229    67
2230    67
2231    67
2232    67
2233    67
2234    67
2235    67
2236    67
2238    67
2225    67
2239    67
2240    67
2241    67
2242    67
2243    67
2244    67
Name: okng_seq, dtype: int64

,group,barcode_information,remark,end_day,end_time,test_time,contents,test_contents,test_ct,from_to_test_ct,okng_seq
0,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.28,3~192.168.108.154~FCT~PPIDBA1WJ25300500277PC3T...,<NA>,NaN,<NA>,<NA>
1,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:57.29,00664~192.168.108.154~FCT~PPIDBA1WJ2530050027...,<NA>,0.01,<NA>,<NA>
2,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.91,START :: DO SET,<NA>,1.62,<NA>,<NA>
3,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.93,DO_SET Value :: 0900800080001000,<NA>,0.02,<NA>,<NA>
4,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.94,테스트 결과 : OK,0.00_d_sig_val_090_set,0.01,1.66,1
...,...,...,...,...,...,...,...,...,...,...,...
495,22,BA1WJ25300500421PC3T-14F014-AC,Non-PD,2025-11-13,17:14:50,17:14:33.06,Power ON :: 1,<NA>,0.08,<NA>,<NA>
496,22,BA1WJ25300500421PC3T-14F014-AC,Non-PD,2025-11-13,17:14:50,17:14:33.08,테스트 결과 : OK,1.06_ps_on,0.02,6.22,23
497,22,BA1WJ25300500421PC3T-14F014-AC,Non-PD,2025-11-13,17:14:50,17:14:33.09,START :: DMM CURRENT NPLC,<NA>,0.01,<NA>,<NA>
498,22,BA1WJ25300500421PC3T-14F014-AC,Non-PD,2025-11-13,17:14:50,17:14:33.11,CURR_NPLC SET :: 0.6,<NA>,0.02,<NA>,<NA>


In [9]:
# ============================================
# Cell 8-1) df4_out → 예지보전 Feature Table 저장
#   schema : e4_predictive_maintenance
#   table  : non_pd_cal_test_ct   (없으면 자동 생성)
# ============================================

import urllib.parse
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "non_pd_cal_test_ct"

def get_engine_remote(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?connect_timeout=5",
        pool_pre_ping=True
    )

engine_pm = get_engine_remote(DB_CONFIG)

# 스키마 생성
with engine_pm.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"))

# 테이블 저장 (없으면 생성, 있으면 append)
df4_out.to_sql(
    TARGET_TABLE,
    engine_pm,
    schema=TARGET_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] Saved → {TARGET_SCHEMA}.{TARGET_TABLE} rows={len(df4_out)}")

[OK] Saved → e4_predictive_maintenance.non_pd_cal_test_ct rows=705851


In [10]:
# ============================================
# Cell 9) 정규화 테이블 (contents/test_ct 제외 + test_contents NA 제외)
# ============================================
df_final = df4.copy()

# test_contents <NA> 제거
df_final = df_final[df_final["test_contents"].notna()].copy()

df_final = df_final[[
    "group",
    "barcode_information",
    "remark",
    "end_day",
    "end_time",
    "test_time",
    "test_contents",
    "from_to_test_ct",
    "okng_seq",
]].reset_index(drop=True)

print(f"[OK] df_final rows={len(df_final)} groups={df_final['group'].nunique()}")
df_final.head(500)

[OK] df_final rows=222512 groups=3332


,group,barcode_information,remark,end_day,end_time,test_time,test_contents,from_to_test_ct,okng_seq
0,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.94,0.00_d_sig_val_090_set,1.66,1
1,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:58.98,0.00_load_a_cc_set,1.7,2
2,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:17:59.63,0.00_dmm_c_rng_set,2.35,3
3,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:18:00.04,0.00_load_c_cc_set,2.76,4
4,20,BA1WJ25300500277PC3T-14F014-AC,Non-PD,2025-11-13,17:18:18,17:18:00.28,0.00_dmm_c_rng_set,3.0,5
...,...,...,...,...,...,...,...,...,...
495,27,BA1WJ25301503674PC3T-14F014-AC,Non-PD,2025-11-13,17:16:59,17:16:42.83,1.08_fw_ver_check,7.61,27
496,27,BA1WJ25301503674PC3T-14F014-AC,Non-PD,2025-11-13,17:16:59,17:16:43.10,1.09_chip_id_check,7.88,28
497,27,BA1WJ25301503674PC3T-14F014-AC,Non-PD,2025-11-13,17:16:59,17:16:43.34,1.10_dmm_3c_rng_set,8.12,29
498,27,BA1WJ25301503674PC3T-14F014-AC,Non-PD,2025-11-13,17:16:59,17:16:43.39,1.10_load_a_5.5c_set,8.17,30


In [11]:
# ============================================
# Cell 10) test_contents × end_day box 통계 + del_out_av + problem1~4(맨 뒤)
# - end_day=2025-11-13 단일
# - 67 step 강제(항상 67행)
# - 소수점 2자리 반올림
# ============================================

import numpy as np
import pandas as pd

TARGET_DAY = TARGET_END_DAY  # '2025-11-13'

work = df_final.copy()
work["end_day"] = pd.to_datetime(work["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
work = work[work["end_day"] == TARGET_DAY].copy()

work["from_to_test_ct"] = pd.to_numeric(work["from_to_test_ct"], errors="coerce")
work = work.dropna(subset=["test_contents", "from_to_test_ct"]).copy()

def box_stats(arr_series: pd.Series) -> dict:
    arr = arr_series.astype(float).to_numpy()
    if arr.size == 0:
        return {
            "lower_outlier": np.nan,
            "1q": np.nan,
            "median": np.nan,
            "3q": np.nan,
            "upper_outlier": np.nan,
            "del_out_av": np.nan,
        }
    q1 = np.nanpercentile(arr, 25)
    med = np.nanpercentile(arr, 50)
    q3 = np.nanpercentile(arr, 75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    arr_in = arr[(arr >= lower) & (arr <= upper)]
    del_out_av = float(np.nanmean(arr_in)) if arr_in.size > 0 else np.nan
    return {
        "lower_outlier": float(lower),
        "1q": float(q1),
        "median": float(med),
        "3q": float(q3),
        "upper_outlier": float(upper),
        "del_out_av": float(del_out_av),
    }

# =========================================================
# problem1~4 매핑 (요청 사양 그대로)
# - 값: (problem1, problem2, problem3, problem4)
# - 비어있으면 None -> 최종 pd.NA로 변환
# =========================================================
PROBLEM4_MAP_67 = {
    "0.00_d_sig_val_090_set": ("relay_board", None, None, None),
    "0.00_load_a_cc_set": ("load_a", "relay_board", None, None),
    "0.00_dmm_c_rng_set": ("dmm", "relay_board", None, None),
    "0.00_load_c_cc_set": ("load_c", "relay_board", None, None),
    "0.00_dmm_regi_set": ("dmm", "relay_board", None, None),
    "0.00_dmm_regi_ac_0.6_set": ("dmm", "relay_board", None, None),

    "1.00_mini_b_short_check": ("mini_b", "dmm", "relay_board", None),
    "1.01_usb_a_short_check": ("usb_a", "dmm", "relay_board", None),
    "1.02_usb_c_short_check": ("usb_c", "dmm", "relay_board", None),

    "1.03_d_sig_val_000_set": ("relay_board", None, None, None),
    "1.03_dmm_regi_set": ("dmm", "relay_board", None, None),
    "1.03_dmm_regi_ac_0.6_set": ("dmm", "relay_board", None, None),

    "1.03_pin12_short_check": ("probe_pin", "dmm", "relay_board", None),
    "1.04_pin23_short_check": ("probe_pin", "dmm", "relay_board", None),
    "1.05_pin34_short_check": ("probe_pin", "dmm", "relay_board", None),

    "1.06_dmm_dc_v_set": ("dmm", "relay_board", None, None),
    "1.06_dmm_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.06_dmm_c_set": ("dmm", "relay_board", None, None),

    "1.06_load_a_sensing_on": ("load_a", "relay_board", None, None),
    "1.06_load_c_sensing_on": ("load_c", "relay_board", None, None),

    "1.06_ps_16.5v_set": ("power_supply", "dmm", "relay_board", None),
    "1.06_ps_on": ("power_supply", "relay_board", None, None),

    "1.06_input_16.5v": ("probe_pin", "power_supply", "dmm", "relay_board"),
    "1.07_idle_c_check": ("probe_pin", "dmm", "relay_board", "power_supply"),

    "1.08_fw_ver_check": ("mini_b B", "linux", "relay_board", None),
    "1.09_chip_id_check": ("mini_b B", "linux", "relay_board", None),

    "1.10_dmm_3c_rng_set": ("dmm", "relay_board", None, None),
    "1.10_load_a_5.5c_set": ("load_a", "dmm", "relay_board", None),
    "1.10_load_a_on": ("load_a", "relay_board", None, None),

    "1.10_usb_a_v_check": ("usb_a", "load_a", "dmm", "relay_board"),
    "1.11_usb_a_c_check": ("usb_a", "load_a", "dmm", "relay_board"),
    "1.12_usb_c_v_check": ("usb_c", "load_a", "dmm", "relay_board"),

    "1.13_load_a_off": ("load_a", "relay_board", None, None),
    "1.13_load_c_5.5c_set": ("load_c", "dmm", "relay_board", None),
    "1.13_load_c_on": ("load_c", "relay_board", None, None),

    "1.13_overcurr_usb_c_v": ("usb_c", "dmm", "relay_board", "load_c"),
    "1.14_overcurr_usb_c_c": ("usb_c", "dmm", "relay_board", "load_c"),

    "1.15_usb_a_v": ("usb_a", "dmm", "relay_board", "load_c"),

    "1.16_load_c_off": ("load_c", "relay_board", None, None),
    "1.16_dut_reset": ("probe_pin", "power_supply", "relay_board", None),

    "1.16_cc_aside_check": ("usb_c", "pd_board", "dmm", "relay_board"),
    "1.17_cc_bside_check": ("usb_c", "pd_board", "dmm", "relay_board"),

    "1.18_load_a_2.4c_set": ("load_a", "dmm", "relay_board", None),
    "1.18_load_c_3c_set": ("load_c", "dmm", "relay_board", None),
    "1.18_load_a_on": ("load_a", "relay_board", None, None),
    "1.18_load_c_on": ("load_c", "relay_board", None, None),

    "1.18_usb_a_v_check": ("usb_a", "dmm", "relay_board", "load_a"),
    "1.19_usb_a_c_check": ("usb_a", "dmm", "relay_board", "load_a"),

    "1.20_usb_c_v_check": ("usb_c", "dmm", "relay_board", "load_c"),
    "1.21_usb_c_c_check": ("usb_c", "dmm", "relay_board", "load_c"),

    "1.22_load_a_off": ("load_a", "relay_board", None, None),
    "1.22_load_c_off": ("load_c", "relay_board", None, None),

    "1.22_usb_c_carplay": ("usb_c", "mini_b", "linux", "relay_board"),
    "1.23_usb_a_carplay": ("usb_a", "mini_b", "linux", "relay_board"),

    "1.24_usb_c_pm1": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.25_usb_c_pm2": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.26_usb_c_pm3": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.27_usb_c_pm4": ("usb_c", "mini_b", "passmark", "relay_board"),

    "1.28_usb_a_pm1": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.29_usb_a_pm2": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.30_usb_a_pm3": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.31_usb_a_pm4": ("usb_a", "mini_b", "passmark", "relay_board"),

    "1.32_dmm_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.32_dmm_c_rng_set": ("dmm", "relay_board", None, None),
    "1.32_dark_curr_check": ("product", "dmm", "relay_board", None),
}

# =========================
# 통계 산출
# =========================
rows = []
for tc, sub in work.groupby("test_contents", sort=False):
    stats = box_stats(sub["from_to_test_ct"])
    p1, p2, p3, p4 = PROBLEM4_MAP_67.get(str(tc), (None, None, None, None))

    rows.append({
        "test_contents": tc,
        "end_day": TARGET_DAY,
        **stats,
        "problem1": p1,
        "problem2": p2,
        "problem3": p3,
        "problem4": p4,
    })

df_box = pd.DataFrame(rows)

# 67 step 강제(항상 67행) + 순서 보존
all_steps = pd.DataFrame({"test_contents": list(MAP_1_67.values())})
df_box = all_steps.merge(df_box, on="test_contents", how="left")
df_box["end_day"] = TARGET_DAY

# 누락 step도 매핑 표준으로 채움 + None -> pd.NA
for i, col in enumerate(["problem1", "problem2", "problem3", "problem4"]):
    df_box[col] = df_box["test_contents"].apply(
        lambda x: PROBLEM4_MAP_67.get(str(x), (None, None, None, None))[i]
    )
    df_box[col] = df_box[col].where(df_box[col].notna(), pd.NA)

# 컬럼 순서 + 반올림 (problem1~4 맨 뒤)
df_box = df_box[[
    "test_contents",
    "end_day",
    "lower_outlier",
    "1q",
    "median",
    "3q",
    "upper_outlier",
    "del_out_av",
    "problem1",
    "problem2",
    "problem3",
    "problem4",
]]

for c in ["lower_outlier","1q","median","3q","upper_outlier","del_out_av"]:
    df_box[c] = df_box[c].round(2)

print(f"[OK] rows={len(df_box)} (항상 67)")
df_box

[OK] rows=67 (항상 67)


,test_contents,end_day,lower_outlier,1q,median,3q,upper_outlier,del_out_av,problem1,problem2,problem3,problem4
0,0.00_d_sig_val_090_set,2025-11-13,1.38,1.83,1.98,2.13,2.58,1.97,relay_board,<NA>,<NA>,<NA>
1,0.00_load_a_cc_set,2025-11-13,1.41,1.92,2.13,2.26,2.77,2.10,load_a,relay_board,<NA>,<NA>
2,0.00_dmm_c_rng_set,2025-11-13,1.62,2.27,2.46,2.70,3.34,2.47,dmm,relay_board,<NA>,<NA>
3,0.00_load_c_cc_set,2025-11-13,1.52,2.21,2.42,2.67,3.36,2.45,load_c,relay_board,<NA>,<NA>
4,0.00_dmm_c_rng_set,2025-11-13,1.62,2.27,2.46,2.70,3.34,2.47,dmm,relay_board,<NA>,<NA>
5,0.00_dmm_regi_set,2025-11-13,1.95,2.67,2.89,3.15,3.87,2.89,dmm,relay_board,<NA>,<NA>
6,0.00_dmm_regi_ac_0.6_set,2025-11-13,2.30,3.02,3.24,3.50,4.22,3.24,dmm,relay_board,<NA>,<NA>
7,1.00_mini_b_short_check,2025-11-13,2.38,3.10,3.32,3.58,4.30,3.32,mini_b,dmm,relay_board,<NA>
8,1.01_usb_a_short_check,2025-11-13,2.46,3.24,3.50,3.76,4.54,3.49,usb_a,dmm,relay_board,<NA>
9,1.02_usb_c_short_check,2025-11-13,2.52,3.38,3.66,3.95,4.81,3.67,usb_c,dmm,relay_board,<NA>


In [12]:
# ============================================
# Cell 10-1) df_box → DB 저장 (summary)
#   schema : e4_predictive_maintenance
#   table  : non_pd_cal_test_ct_summary (없으면 생성)
# ============================================

import urllib.parse
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "non_pd_cal_test_ct_summary"

def get_engine_remote(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?connect_timeout=5",
        pool_pre_ping=True
    )

engine_pm = get_engine_remote(DB_CONFIG)

# 스키마 생성
with engine_pm.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"))

# 저장(없으면 생성, 있으면 append)
df_box.to_sql(
    TARGET_TABLE,
    engine_pm,
    schema=TARGET_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] Saved → {TARGET_SCHEMA}.{TARGET_TABLE} rows={len(df_box)} end_day={TARGET_DAY}")

[OK] Saved → e4_predictive_maintenance.non_pd_cal_test_ct_summary rows=67 end_day=2025-11-13
